## Gaussian Mixture Model / ガウス混合モデル

モデルがわかっている場合には、その関数でフィットして、パラメータを決めていくだけです。ここでは、よくある例として、ピークが複数ある分布を、複数のガウシアンの足しあわせで近似する場合を考えます。

When the model is known, we simply fit that function and determine the parameters. Here, as a common example, we consider the case of approximating a distribution with multiple peaks as a sum of multiple Gaussians.

### 1 次元 / 1-Dimensional


In [ ]:
! wget https://raw.githubusercontent.com/vitroid/PythonTutorials/master/2%20Advanced/data7.txt

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# data7.txtは1カラムのデータ
file = open("data7.txt", "r")
xL = []
for line in file:
    cols = line.split()
    x = float(cols[0])
    xL.append(x)

xs = np.array(xL)
X = xs[:, np.newaxis]  # 列ベクトルに変換
plt.hist(
    X, 30, density=True, histtype="stepfilled", alpha=0.4
)  # ヒストグラムでプロット

3 つのガウシアン分布でこの分布をフィットしてみる。(http://scikit-learn.org/stable/modules/generated/sklearn.mixture.GMM.html)

Let's try to fit this distribution with three Gaussian distributions.


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# 3つのガウス関数でフィットする。
model = GaussianMixture(3).fit(X)

x = np.linspace(-6, 6, 1000)
x = x[:, np.newaxis]  # 列ベクトルに変換
y = np.exp(
    model.score_samples(x)
)  # 推定された分布関数(scoreは対数を返すので、指数関数をかけている)
plt.hist(X, 30, density=True, histtype="stepfilled", alpha=0.4)
plt.plot(x, y)

3 つの成分に分けてみます。

Let's separate it into three components.


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# 3つのガウス関数でフィットする。
model = GaussianMixture(3).fit(X)

x = np.linspace(-6, 6, 1000)
x = x[:, np.newaxis]  # 列ベクトルに変換
y = np.exp(model.score_samples(x))  # 推定された分布関数
w = model.predict_proba(x)  # 各Gaussianの"重み"
y0 = w[:, 0] * y
y1 = w[:, 1] * y
y2 = w[:, 2] * y

plt.hist(X, 30, density=True, histtype="stepfilled", alpha=0.4)
plt.plot(x, y)
plt.plot(x, y0)
plt.plot(x, y1)
plt.plot(x, y2)

重みだけをプロットしてみます。

Let's plot only the weights.


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# 3つのガウス関数でフィットする。
model = GaussianMixture(3).fit(X)

x = np.linspace(-6, 6, 1000)
x = x[:, np.newaxis]  # 列ベクトルに変換
y = np.exp(model.score(x))  # 推定された分布関数
w = model.predict_proba(x)  # 各Gaussianの"重み"

plt.plot(x, w)

このグラフは何を意味しているのでしょうか。

What does this graph mean?

実験で得られた分布が、仮に 3 つのガウシアンの足しあわせでできているとしましょう。

Let's assume that the distribution obtained from the experiment is made up of a sum of three Gaussians.

それはつまりこういうことです。測定している系は 3 つの独立な状態のいずれかに属しています。そして、状態 1 では、測定値は平均値-1 のまわりで広い分布をもち、状態 2 では平均値 0 で分布し、状態 3 では平均値 3 のまわりで分布します。

This means that the system being measured belongs to one of three independent states. In state 1, the measured values have a broad distribution around a mean of -1, in state 2 they are distributed around a mean of 0, and in state 3 they are distributed around a mean of 3.

測定する人間には、測定結果しかわからないので、系がどの状態にあるのかはわかりません。

To the person making the measurement, only the measurement results are known, so they don't know which state the system is in.

もし、測定値が 1 だった時、系はどの状態にあると推定するのが一番合理的でしょうか。

If the measured value was 1, which state would be most rational to estimate the system is in?

もちろん、状態 2 にあると推定すべきでしょう。なぜなら、x=1 では、3 つの状態のなかで、状態 2 が一番確率が高いからです。これが賭け事だとすれば、あなたは状態 2 に賭けるべきなのは明らかです。

Of course, you should estimate that it's in state 2. This is because at x=1, state 2 has the highest probability among the three states. If this were gambling, you should obviously bet on state 2.

上のグラフの上側エンベロープは、あなたが賭けるべき状態を示しています。あるいは、一番もっともらしい、データの分類方法を示しているとも言えます。また、線が交わる点は、いちばんもっともらしい状態が入れかわる、「決定境界」を示しています。

The upper envelope of the graph above shows the state you should bet on. Or it can be said to show the most plausible way to classify the data. The points where the lines intersect show the "decision boundaries" where the most plausible state changes.

ただし、一つ心配な点があります。この分布は、本当に 3 つのガウシアン関数の足しあわせで良いのでしょうか。

However, there is one concern. Is this distribution really well represented by the sum of three Gaussian functions?

試しに、2 つのガウシアンでフィットすると、あまり良くないことはわかります。

As a test, if we fit with two Gaussians, we can see that it's not very good.


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# 2つのガウス関数でフィットする。
model = GaussianMixture(2).fit(X)

x = np.linspace(-6, 6, 1000)
x = x[:, np.newaxis]  # 列ベクトルに変換
y = np.exp(
    model.score_samples(x)
)  # 推定された分布関数(scoreは対数を返すので、指数関数をかけている)
plt.hist(X, 30, density=True, histtype="stepfilled", alpha=0.4)
plt.plot(x, y)

でも、4 つのガウシアンでも問題ないようです。

But four Gaussians also seem to work fine.


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# 4つのガウス関数でフィットする。
model = GaussianMixture(4).fit(X)

x = np.linspace(-6, 6, 1000)
x = x[:, np.newaxis]  # 列ベクトルに変換
y = np.exp(
    model.score_samples(x)
)  # 推定された分布関数(scoreは対数を返すので、指数関数をかけている)
plt.hist(X, 30, density=True, histtype="stepfilled", alpha=0.4)
plt.plot(x, y)

では、最適な個数はいくつでしょうか。ここでも、あらかじめ 3 つの状態しかないことがわかっていれば良いのですが、それがわからない場合には困ってしまいます。ガウシアンの個数を増やせば、より忠実にヒストグラムのでこぼこに追従するようになりますが、それは本当に知りたい答とは限りません。

So what is the optimal number? Here too, it would be good if we knew in advance that there are only three states, but if we don't know that, we're in trouble. If we increase the number of Gaussians, they will follow the irregularities of the histogram more faithfully, but that's not necessarily the answer we really want to know.

一つの方法として、情報量規準を使う評価法があります。情報量規準は、統計モデルの良さを評価する指標です。一般に、モデルの次数を上げるほど、データとモデルの適合は良くなるが、過学習(過適合)に陥りがちになります。情報量規準はモデルの複雑さと、データの適合度のバランスをとるための「考え方」で、AIC, BIC など、いくつもの規準が提案されています。

One method is to use information criteria for evaluation. Information criteria are measures for evaluating the goodness of statistical models. Generally, the higher the model order, the better the fit between data and model becomes, but it tends to fall into overfitting. Information criteria are "approaches" for balancing model complexity and data fit, and several criteria such as AIC and BIC have been proposed.

GaussianMixture には AIC(赤池情報量規準)と BIC(ベイズ情報量規準)があらかじめ準備されていますので、両方で最適なガウス関数の数を決めてみましょう。

GaussianMixture has AIC (Akaike Information Criterion) and BIC (Bayesian Information Criterion) prepared in advance, so let's determine the optimal number of Gaussian functions using both.


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

AIC = np.zeros(10)
BIC = np.zeros(10)
for n in range(1, 10):
    model = GaussianMixture(n).fit(X)
    AIC[n] = model.aic(X)
    BIC[n] = model.bic(X)

plt.plot(AIC, label="AIC")
plt.plot(BIC, label="BIC")
plt.xlim(1, 9)
plt.ylim(3800, 4100)
plt.legend()

評価は分かれました。AIC は 3 つのガウシアンで十分だ、BIC は、いや 2 でも 3 でも大差ない、という答になりました。いずれにしても、4 つ以上のガウス関数を使う必要はなさそうです。

The evaluations were divided. AIC says three Gaussians are sufficient, while BIC says there's not much difference between 2 or 3. In any case, it seems there's no need to use four or more Gaussian functions.


### 2 次元以上 / 2D and Higher Dimensions

2 次元以上の場合でも、ガウス分布の足しあわせに分解できます。

Even in the case of two or more dimensions, it can be decomposed into a sum of Gaussian distributions.

https://github.com/scikit-learn/scikit-learn/blob/master/examples/mixture/plot_gmm_pdf.py


In [ ]:
%matplotlib inline
#Modified from:
#https://github.com/scikit-learn/scikit-learn/blob/master/examples/mixture/plot_gmm_pdf.py

#サンプルデータの生成
import numpy as np
import matplotlib.pyplot as plt

n_samples = 300

# generate random sample, two components
np.random.seed(0)

# generate spherical data centered on (20, 20)
shifted_gaussian = np.random.randn(n_samples, 2) + np.array([20, 20])

# generate zero centered stretched Gaussian data
C = np.array([[0., -0.7], [3.5, .7]])
stretched_gaussian = np.dot(np.random.randn(n_samples, 2), C)

# concatenate the two datasets into the final training set
X_train = np.vstack([shifted_gaussian, stretched_gaussian])

plt.scatter(X_train[:, 0], X_train[:, 1], .8)

plt.axis('tight')
plt.show()

In [ ]:
# 複数の二次元ガウス関数でフィットする。
from matplotlib.colors import LogNorm
from sklearn import mixture

# fit a Gaussian Mixture Model with two components
clf = mixture.GaussianMixture(n_components=2, covariance_type="full")
clf.fit(X_train)

# display predicted scores by the model as a contour plot
x = np.linspace(-20.0, 30.0)
y = np.linspace(-20.0, 40.0)
X, Y = np.meshgrid(x, y)
XX = np.array([X.ravel(), Y.ravel()]).T
Z = -clf.score_samples(XX)
Z = Z.reshape(X.shape)

CS = plt.contour(
    X, Y, Z, norm=LogNorm(vmin=1.0, vmax=1000.0), levels=np.logspace(0, 3, 10)
)
CB = plt.colorbar(CS, shrink=0.8, extend="both")
plt.scatter(X_train[:, 0], X_train[:, 1], 0.8)

plt.title("Negative log-likelihood predicted by a GMM")
plt.axis("tight")
plt.show()